# AI purchasing recommendations with Gemini

The inventory notebook flagged products at risk of running out or holding too much stock, but a table of
numbers isn't easy for a busy store manager to act on. Here we send the flagged products to Google's
**Gemini** AI model and ask it to write a clear recommendation for each one: what to do, how urgent it is,
and why.

Two rules keep the AI honest:
1. It may only use the numbers we give it and must not invent anything.
2. The final table puts the AI's recommendation **next to the real numbers**, so every recommendation can
   be checked against the data.

In [19]:
# Install the libraries this notebook needs (only needed once)
%pip install google-generativeai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Step 1: Set up the Gemini client

- **`google.generativeai`** is Google's Python library for talking to Gemini.
- **`dotenv`** reads secret settings from the `.env` file in the project folder. Keeping the API key in
  `.env` (which is listed in `.gitignore`) means it never appears in the notebook and never gets shared by
  accident.
- **`os.getenv()`** reads a setting once it's loaded.

**About the model name:** Google retires older models over time. `gemini-2.0-flash` has been retired, and
requests to it now fail with a "404 ... is no longer available" error. We use `gemini-3.6-flash`, the
replacement Google recommends. If this model is retired later, change `MODEL_NAME` below.

In [20]:
import json
import os
import time

import pandas as pd
import google.generativeai as genai
from dotenv import load_dotenv
from google.api_core.exceptions import ResourceExhausted  # the error Gemini raises for "429: rate limited"

# Read the .env file in the project folder (one level up from notebooks/) and load its settings
load_dotenv("../.env")
api_key = os.getenv("GEMINI_API_KEY")

# Fail early with a clear message if the key wasn't found, rather than a confusing error later
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Check that the .env file exists in the project folder.")

# Give the library our API key; every request below uses it automatically
genai.configure(api_key=api_key)

MODEL_NAME = "gemini-3.6-flash"
model = genai.GenerativeModel(MODEL_NAME)

# model.model_name is the exact name the library sends to Google with every request
print("Gemini client ready, calling model:", model.model_name)

Gemini client ready, calling model: models/gemini-3.6-flash


## Step 2: Load the inventory table and keep only the flagged products

Products with no stockout risk and no overstock don't need a recommendation, so we only send the flagged
ones. That also means fewer requests to the API.

In [21]:
inventory_df = pd.read_csv("../data/inventory_status.csv")

# | means OR: keep rows where EITHER flag is True
flagged_df = inventory_df[inventory_df["stockout_risk"] | inventory_df["overstock_flag"]].reset_index(drop=True)

print(f"Flagged products: {len(flagged_df)} of {len(inventory_df)}")
print(f"  Stockout risk: {flagged_df['stockout_risk'].sum()}")
print(f"  Overstocked:   {flagged_df['overstock_flag'].sum()}")

Flagged products: 72 of 500
  Stockout risk: 61
  Overstocked:   11


### Load recommendations we already have (checkpoint)

The free tier allows only **20 requests per day**, so a run can be cut short part-way through. To avoid
losing work, every successful batch is saved straight away to `data/ai_recommendations_partial.csv`
(Step 5). This file is our **checkpoint**: a record of the work already done.

Here we read that file if it exists. Any product already in it is skipped in Step 5, so we never ask Gemini
again for a recommendation we already have. After a quota cut-off, you simply re-run the notebook the next
day and it carries on where it stopped.

To start completely fresh, delete `data/ai_recommendations_partial.csv`.

In [22]:
PARTIAL_PATH = "../data/ai_recommendations_partial.csv"
RECOMMENDATION_COLUMNS = ["store", "item", "action", "urgency", "reason"]

# os.path.exists() checks whether a file is there
if os.path.exists(PARTIAL_PATH):
    saved_df = pd.read_csv(PARTIAL_PATH)
    # .to_dict(orient="records") turns the table back into a list of dictionaries,
    # the same shape Gemini's answers have, so old and new results can go in one list
    saved_recommendations = saved_df.to_dict(orient="records")
    print(f"Found checkpoint file with {len(saved_recommendations)} saved recommendations.")
else:
    saved_recommendations = []
    print("No checkpoint file yet: starting from scratch.")

# The (store, item) pairs we already have, as a set (a collection with no duplicates, fast to search)
done_products = {(int(rec["store"]), int(rec["item"])) for rec in saved_recommendations}

Found checkpoint file with 72 saved recommendations.


## Step 3: Build the prompt

A **prompt** is the text instruction we send to the AI. A good prompt is very specific about:
- **its role**: an inventory planning assistant;
- **the data**: the flagged products, written as JSON (a standard text format for structured data);
- **the rules**: especially "use only the numbers provided";
- **the output format**: *only* a JSON array, so our code can read the answer reliably.

**Urgency rules.** Urgency is **not** left to the AI's judgment. It's pure arithmetic on numbers we
already have, so we give the AI exact thresholds and tell it to apply them and nothing else. The
thresholds are defined below and easy to change:
- **Stock-out risk** (a delivery takes 7 days):
  - High: fewer than 5.5 days of stock left
  - Medium: 5.5 to under 6.5 days
  - Low: 6.5 days or more (but still flagged)
- **Overstock**, measured against the demand forecast for the next 14 days:
  - High: stock is more than 2x the 14-day demand (more than 28 days of stock)
  - Medium: stock is 1.5x to 2x the 14-day demand (21 to 28 days)
  - Low: otherwise

Because urgency is just arithmetic, the final urgency is also **recalculated in Python** at the end of the
notebook (Step 10). That way the saved table never depends on the AI getting the maths right.


In [23]:
# Urgency thresholds (change these to adjust the rules)
# Stock-out risk, in days of stock left:
STOCKOUT_HIGH_BELOW = 5.5    # < 5.5 days        -> High
STOCKOUT_MEDIUM_BELOW = 6.5  # 5.5 to < 6.5 days -> Medium, 6.5 days or more -> Low
# Overstock, as a multiple of the next 14 days' predicted demand:
FORECAST_DAYS = 14
OVERSTOCK_HIGH_RATIO = 2.0   # stock > 2x the 14-day demand         -> High
OVERSTOCK_MEDIUM_RATIO = 1.5 # stock 1.5x to 2x the 14-day demand   -> Medium, otherwise -> Low

# The columns we send to the AI: only the numbers it needs, nothing else
PROMPT_COLUMNS = ["store", "item", "current_stock", "avg_daily_predicted_demand",
                  "days_of_stock_left", "stockout_risk", "reorder_qty", "overstock_flag"]


def build_prompt(batch_df):
    """Build the instruction text for one batch of products."""
    # .to_dict(orient="records") turns the table into a list with one dictionary per row, e.g.
    #   [{"store": 5, "item": 39, "current_stock": 141, ...}, ...]
    # json.dumps() then turns that list into JSON text we can paste into the prompt.
    products_json = json.dumps(batch_df[PROMPT_COLUMNS].to_dict(orient="records"), indent=2)

    # An f-string (f"...") fills in the {variables}. Literal curly braces in the JSON example
    # below are written as {{ and }} so Python doesn't treat them as variables.
    return f"""You are an inventory planning assistant for a retail chain.

Below is a list of products that need attention. Each product has:
- store, item: which product it is
- current_stock: units currently in stock
- avg_daily_predicted_demand: forecast units sold per day over the next 14 days
- days_of_stock_left: how many days the current stock will last at that rate
- stockout_risk: true if stock will run out before a new order (7-day delivery time) can arrive
- reorder_qty: the calculated number of units to order now
- overstock_flag: true if there is far more stock than the next 14 days need

Products:
{products_json}

For EACH product, give one recommendation:
- action: "Reorder now" if stockout_risk is true, "Reduce future orders" if overstock_flag is true
- urgency: "High", "Medium" or "Low". Apply these exact thresholds using the numeric values given —
  do not use your own judgment for urgency, only these rules:
  - For stockout-risk products (action "Reorder now"), using days_of_stock_left:
    - High: days_of_stock_left < {STOCKOUT_HIGH_BELOW}
    - Medium: days_of_stock_left is {STOCKOUT_HIGH_BELOW} or more but less than {STOCKOUT_MEDIUM_BELOW}
    - Low: days_of_stock_left is {STOCKOUT_MEDIUM_BELOW} or more
  - For overstock products (action "Reduce future orders"), first calculate
    next_{FORECAST_DAYS}_days_demand = avg_daily_predicted_demand x {FORECAST_DAYS}. Then:
    - High: current_stock > {OVERSTOCK_HIGH_RATIO} x next_{FORECAST_DAYS}_days_demand
    - Medium: current_stock is between {OVERSTOCK_MEDIUM_RATIO} x and {OVERSTOCK_HIGH_RATIO} x next_{FORECAST_DAYS}_days_demand
    - Low: otherwise
- reason: ONE sentence explaining why, quoting the relevant numbers. For "Reorder now", mention reorder_qty.

Use only the numbers provided. Do not invent or assume any information not given.

Return ONLY a JSON array, with no other text before or after it. One object per product, in this format:
[
  {{"store": 5, "item": 39, "action": "Reorder now", "urgency": "Medium", "reason": "..."}}
]
"""


# Preview the prompt for the first 2 products, to see exactly what the AI receives
print(build_prompt(flagged_df.head(2)))

You are an inventory planning assistant for a retail chain.

Below is a list of products that need attention. Each product has:
- store, item: which product it is
- current_stock: units currently in stock
- avg_daily_predicted_demand: forecast units sold per day over the next 14 days
- days_of_stock_left: how many days the current stock will last at that rate
- stockout_risk: true if stock will run out before a new order (7-day delivery time) can arrive
- reorder_qty: the calculated number of units to order now
- overstock_flag: true if there is far more stock than the next 14 days need

Products:
[
  {
    "store": 5,
    "item": 39,
    "current_stock": 141,
    "avg_daily_predicted_demand": 27.7,
    "days_of_stock_left": 5.1,
    "stockout_risk": true,
    "reorder_qty": 139,
    "overstock_flag": false
  },
  {
    "store": 9,
    "item": 23,
    "current_stock": 132,
    "avg_daily_predicted_demand": 25.9,
    "days_of_stock_left": 5.1,
    "stockout_risk": true,
    "reorder_qty

## Step 4: Call Gemini and read its answer

`get_recommendations()` sends one batch to Gemini and turns the answer into Python data.

Some new ideas:
- **`json.loads()`** does the opposite of `json.dumps()`: it reads JSON *text* and turns it into Python
  lists and dictionaries we can work with.
- **Code fences.** AI models often wrap JSON in markdown like ` ```json ... ``` `. Those extra characters
  would make `json.loads()` fail, so we strip them first.
- **`try` / `except`.** Normally, if something goes wrong (the network drops, or the AI returns broken
  JSON), Python stops the whole notebook with an error. Code inside `try:` is attempted; if it fails, Python
  jumps to `except:` instead of crashing. We print a warning and return an empty list, so one bad batch
  doesn't lose the results of all the others.

**Rate limits.** The free tier has two limits for this model, and going over either one makes Gemini refuse
the request with error **429** (`ResourceExhausted` in Python). We handle them differently:
- **Per-minute limit (5 requests per minute):** this clears within seconds, so we **wait 20 seconds and try
  again**, up to `MAX_ATTEMPTS` (3) attempts in total.
- **Per-day limit (20 requests per day):** this doesn't clear until midnight Pacific time, so retrying just
  wastes time. The error message then contains `PerDay`. When we see that, we stop **all** remaining
  batches straight away. Everything done so far is already saved in the checkpoint file.

Other errors, such as broken JSON, won't be fixed by waiting, so those are not retried.

**A custom error.** `class DailyQuotaExhausted(Exception)` creates our own error type. Raising it inside
the function and catching it in the batch loop is how the function tells the loop "stop everything",
which is different from "this one batch failed".

In [24]:
MAX_ATTEMPTS = 3          # 1 first try + up to 2 retries
RETRY_WAIT_SECONDS = 20   # how long to wait after a per-minute rate-limit error


class DailyQuotaExhausted(Exception):
    """Our own error type, raised when the daily quota is used up, so the batch loop knows to stop."""


def call_gemini_with_retry(prompt):
    """Send the prompt to Gemini, retrying if we are rate limited. Returns the response text."""
    # range(1, MAX_ATTEMPTS + 1) counts the attempts: 1, 2, 3
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            # response_mime_type="application/json" asks Gemini to reply in pure JSON,
            # which makes code fences less likely (we still strip them later, just in case)
            response = model.generate_content(
                prompt,
                generation_config={"response_mime_type": "application/json"},
            )
            return response.text  # success: "return" leaves the function (and the loop) immediately

        except ResourceExhausted as error:
            # This except only catches rate-limit errors. Anything else passes straight up to the caller.

            # Daily limit: waiting won't help, so give up immediately with our own error type.
            # str(error) is the error message as text; "in" checks whether it contains "PerDay".
            if "PerDay" in str(error):
                raise DailyQuotaExhausted(str(error)) from error

            # Per-minute limit: wait and try again, unless we've used up our attempts
            if attempt == MAX_ATTEMPTS:
                raise  # out of attempts: "raise" passes the error on, so the caller's except handles it
            print(f"  Rate limited, waiting {RETRY_WAIT_SECONDS}s before retry {attempt} of {MAX_ATTEMPTS - 1}")
            time.sleep(RETRY_WAIT_SECONDS)


def get_recommendations(batch_df):
    """Send one batch to Gemini and return its recommendations as a list of dictionaries."""
    prompt = build_prompt(batch_df)

    try:
        raw_response_text = call_gemini_with_retry(prompt)

        # DEBUG: show exactly what Gemini sent back, before we clean or parse it
        print("  --- Raw response from Gemini ---")
        print(raw_response_text)
        print("  --- End of raw response ---")

        response_text = raw_response_text.strip()

        # Strip markdown code fences if present: remove a leading ```json (or ```) and a trailing ```
        if response_text.startswith("```"):
            response_text = response_text.removeprefix("```json").removeprefix("```")
            response_text = response_text.removesuffix("```").strip()

        # Turn the JSON text into a Python list of dictionaries
        recommendations = json.loads(response_text)

        # Warn (but continue) if the AI skipped some products
        if len(recommendations) != len(batch_df):
            print(f"  Warning: sent {len(batch_df)} products but got {len(recommendations)} recommendations back.")
        return recommendations

    except DailyQuotaExhausted:
        # Don't swallow this one: "raise" passes it on to the batch loop, which stops all batches.
        # (This except must come BEFORE "except Exception", which would otherwise catch it first.)
        raise

    except Exception as error:
        # Any failure (network, API error, broken JSON) lands here instead of crashing the notebook
        print(f"  Warning: this batch failed and was skipped ({type(error).__name__}: {error})")
        return []

## Step 5: Send the products in batches

We send **15 products per request**. With 72 products that's only 5 requests, well inside the free tier's
20 requests per day.

**`time.sleep(15)`** pauses for 15 seconds between requests, to stay under the 5-requests-per-minute limit.
The retry from Step 4 covers the occasional time we still hit it.

Three more things happen in this cell:
- **Skipping finished products.** Products already in the checkpoint file are left out before batching.
- **Saving as we go.** After each successful batch, its results are **appended** to the checkpoint file
  straight away. `mode="a"` adds rows to the end of the file instead of replacing it, and the column
  headers are only written when the file is first created.
- **Stopping on the daily limit.** If Step 4 reports the daily quota is used up, **`break`** exits the
  batch loop immediately, and the remaining batches are left for the next run.

At the end we report how many batches succeeded, failed or were not attempted, and list any products still
missing a recommendation.

In [25]:
BATCH_SIZE = 15
PAUSE_BETWEEN_BATCHES_SECONDS = 15

# Leave out products already in the checkpoint file.
# zip() pairs up the store and item columns row by row; "not in" keeps only pairs we haven't done yet.
is_not_done = [(store, item) not in done_products for store, item in zip(flagged_df["store"], flagged_df["item"])]
remaining_df = flagged_df[is_not_done].reset_index(drop=True)
print(f"Skipped (already done, loaded from checkpoint): {len(flagged_df) - len(remaining_df)} products")
print(f"Still to request: {len(remaining_df)} products")

# range(0, n, 15) gives the starting row of each batch: 0, 15, 30, ...
# .iloc[start : start + 15] then takes 15 rows from that starting point.
batches = [remaining_df.iloc[start:start + BATCH_SIZE] for start in range(0, len(remaining_df), BATCH_SIZE)]
print(f"Sending {len(remaining_df)} products in {len(batches)} batches to model: {model.model_name}\n")

# Start from the recommendations we already had, then add new ones as batches succeed
all_recommendations = list(saved_recommendations)
num_succeeded = 0
num_failed = 0
num_not_attempted = 0

for batch_number, batch_df in enumerate(batches, start=1):  # enumerate counts the batches: 1, 2, 3, ...
    try:
        batch_results = get_recommendations(batch_df)
    except DailyQuotaExhausted:
        # Every batch from this one onwards is left for the next run
        num_not_attempted = len(batches) - batch_number + 1
        print(f"Daily quota exhausted, stopping. {num_not_attempted} batch(es) not attempted; "
              "re-run after the quota resets (midnight Pacific time) to continue.")
        break  # "break" exits the for loop immediately

    # get_recommendations returns an empty list when a batch fails
    if len(batch_results) > 0:
        num_succeeded += 1  # += 1 means "add 1 to this counter"
        all_recommendations.extend(batch_results)  # .extend() adds every item of one list to another

        # Checkpoint: append this batch's results to the file right away.
        # .reindex(columns=...) keeps exactly our 5 columns, in a fixed order, so every batch lines up.
        # header=... writes the column names only if the file doesn't exist yet.
        batch_results_df = pd.DataFrame(batch_results).reindex(columns=RECOMMENDATION_COLUMNS)
        batch_results_df.to_csv(PARTIAL_PATH, mode="a", header=not os.path.exists(PARTIAL_PATH), index=False)
    else:
        num_failed += 1
    print(f"Processed batch {batch_number} of {len(batches)} ({len(batch_results)} recommendations)")

    # Pause between requests (no need to wait after the last one)
    if batch_number < len(batches):
        time.sleep(PAUSE_BETWEEN_BATCHES_SECONDS)

print(f"\nBatches succeeded: {num_succeeded} | Batches failed: {num_failed} | Not attempted: {num_not_attempted}")
print(f"Recommendations available (saved + new): {len(all_recommendations)} of {len(flagged_df)} flagged products")

# Which flagged products have no recommendation? Build a set (a collection with no duplicates)
# of (store, item) pairs for each side, then subtract: flagged minus answered = missing.
flagged_products = set(zip(flagged_df["store"], flagged_df["item"]))
answered_products = {(rec.get("store"), rec.get("item")) for rec in all_recommendations}
missing_products = sorted(flagged_products - answered_products)

if missing_products:
    print(f"\n{len(missing_products)} products are missing recommendations (re-run or investigate these):")
    for store, item in missing_products:
        print(f"  Store {store}, item {item}")
else:
    print("Every flagged product has a recommendation.")

Skipped (already done, loaded from checkpoint): 72 products
Still to request: 0 products
Sending 0 products in 0 batches to model: models/gemini-3.6-flash


Batches succeeded: 0 | Batches failed: 0 | Not attempted: 0
Recommendations available (saved + new): 72 of 72 flagged products
Every flagged product has a recommendation.


## Debug: what's actually in `all_recommendations`?

Before building a table from the results, we look at them directly: how many there are, what the first
few look like, and whether each one is a dictionary with the keys we expect (`store`, `item`, `action`,
`urgency`, `reason`).

If the list is empty, `pd.DataFrame([])` in the next step creates a table with **no columns**, so asking
for the `"urgency"` column fails with `KeyError: 'urgency'`.

`repr()` shows a value exactly as Python stores it, including quotes and brackets, so nothing is hidden.

In [26]:
print("Total items in all_recommendations:", len(all_recommendations))

if len(all_recommendations) == 0:
    print("all_recommendations is empty — the batch loop is not returning any successful results. "
          "Re-check the batch loop's error handling and rate limit retry logic.")
else:
    # Show the first 3 items exactly as they are
    print("\nFirst 3 items:")
    for item in all_recommendations[:3]:  # [:3] takes the first 3 items of a list
        print(repr(item))

    # type() tells us what kind of value something is: dict, list, str, ...
    first_item = all_recommendations[0]
    print("\nType of the first item:", type(first_item).__name__)
    if isinstance(first_item, dict):  # isinstance() checks whether a value is of a given type
        print("Keys of the first item:", list(first_item.keys()))
    else:
        print(f"WARNING: the first item is a {type(first_item).__name__}, not a dict. "
              "The next step expects a list of dicts, so it will not work correctly.")

Total items in all_recommendations: 72

First 3 items:
{'store': 5, 'item': 39, 'action': 'Reorder now', 'urgency': 'High', 'reason': 'With only 5.1 days of stock left and an average daily demand of 27.7 units, a reorder of 139 units is required to avoid a stockout.'}
{'store': 9, 'item': 23, 'action': 'Reorder now', 'urgency': 'High', 'reason': 'Current stock of 132 units provides only 5.1 days of stock left, requiring a reorder of 127 units now to prevent running out.'}
{'store': 5, 'item': 27, 'action': 'Reorder now', 'urgency': 'High', 'reason': 'There are only 5.4 days of stock left based on current daily demand, so you should reorder 66 units now.'}

Type of the first item: dict
Keys of the first item: ['store', 'item', 'action', 'urgency', 'reason']


## Step 6: Combine the recommendations into one table, most urgent first

Sorting the text "High", "Medium", "Low" alphabetically would give High, Low, Medium, which is wrong.
A **categorical** column tells pandas the correct order, so sorting puts High first.

In [27]:
recommendations_df = pd.DataFrame(all_recommendations)

# pd.Categorical with ordered=True defines a custom sort order for the urgency column
recommendations_df["urgency"] = pd.Categorical(
    recommendations_df["urgency"], categories=["High", "Medium", "Low"], ordered=True
)
recommendations_df = recommendations_df.sort_values("urgency").reset_index(drop=True)

# How many of each urgency level? .value_counts() counts each distinct value
print(recommendations_df["urgency"].value_counts().to_string())
recommendations_df.head()

urgency
Medium    40
Low       26
High       6


,store,item,action,urgency,reason
0,5,39,Reorder now,High,With only 5.1 days of stock left and an averag...
1,9,23,Reorder now,High,Current stock of 132 units provides only 5.1 d...
2,5,27,Reorder now,High,There are only 5.4 days of stock left based on...
3,3,36,Reorder now,High,Stock will run out in 5.4 days at a daily pred...
4,6,20,Reorder now,High,With 169 units in stock offering only 5.4 days...


## Step 7: Put the real numbers next to each recommendation

We **merge** the AI's recommendations with the original inventory table, matching on `store` and `item`.
The final table then shows what the AI said *and* the numbers behind it, so anyone can check a
recommendation against the data.

We also check automatically that the AI's **action** matches the flags
(stockout -> "Reorder now", overstock -> "Reduce future orders").
Missing recommendations were already reported at the end of Step 5.

In [28]:
NUMBER_COLUMNS = ["store", "item", "current_stock", "avg_daily_predicted_demand",
                  "days_of_stock_left", "reorder_qty", "stockout_risk", "overstock_flag"]

# how="left" keeps every AI recommendation, and adds the matching numbers from flagged_df
final_df = recommendations_df.merge(flagged_df[NUMBER_COLUMNS], on=["store", "item"], how="left")

# Put the most urgent first, and within the same urgency, the fewest days of stock first
final_df = final_df.sort_values(["urgency", "days_of_stock_left"]).reset_index(drop=True)

# Check: the action should follow directly from the flags.
# .map() swaps each value using the dictionary: True -> "Reorder now", False -> "Reduce future orders"
expected_action = final_df["stockout_risk"].map({True: "Reorder now", False: "Reduce future orders"})
num_wrong_action = (final_df["action"] != expected_action).sum()
print("Recommendations whose action doesn't match the flags:", num_wrong_action)

# Final column order: the AI's recommendation first, then the numbers to verify it with
final_df = final_df[["store", "item", "action", "urgency", "reason",
                     "days_of_stock_left", "reorder_qty", "current_stock", "avg_daily_predicted_demand"]]
final_df.head()

Recommendations whose action doesn't match the flags: 0


,store,item,action,urgency,reason,days_of_stock_left,reorder_qty,current_stock,avg_daily_predicted_demand
0,5,39,Reorder now,High,With only 5.1 days of stock left and an averag...,5.1,139,141,27.7
1,9,23,Reorder now,High,Current stock of 132 units provides only 5.1 d...,5.1,127,132,25.9
2,5,27,Reorder now,High,There are only 5.4 days of stock left based on...,5.4,66,75,14.0
3,3,36,Reorder now,High,Stock will run out in 5.4 days at a daily pred...,5.4,340,393,72.6
4,6,20,Reorder now,High,With 169 units in stock offering only 5.4 days...,5.4,145,169,31.2


## Step 8: Save the final table

In [29]:
output_path = "../data/ai_recommendations.csv"
final_df.to_csv(output_path, index=False)  # index=False leaves out pandas' row numbers
print(f"Saved {len(final_df)} recommendations to {output_path}")

Saved 72 recommendations to ../data/ai_recommendations.csv


## Step 9: Review the top 10

The 10 most urgent recommendations, printed one block per product so the full reason is readable.
Check a few of them: do the numbers quoted in the reason match the numbers underneath?

In [30]:
# .head(10) takes the first 10 rows; .itertuples() loops over them one row at a time
for row in final_df.head(10).itertuples():
    print(f"Store {row.store}, item {row.item}  |  {row.action}  |  Urgency: {row.urgency}")
    print(f"  Reason: {row.reason}")
    print(f"  Data:   {row.days_of_stock_left} days of stock left, reorder_qty {row.reorder_qty}, "
          f"current_stock {row.current_stock}, forecast {row.avg_daily_predicted_demand}/day")
    print()

Store 5, item 39  |  Reorder now  |  Urgency: High
  Reason: With only 5.1 days of stock left and an average daily demand of 27.7 units, a reorder of 139 units is required to avoid a stockout.
  Data:   5.1 days of stock left, reorder_qty 139, current_stock 141, forecast 27.7/day

Store 9, item 23  |  Reorder now  |  Urgency: High
  Reason: Current stock of 132 units provides only 5.1 days of stock left, requiring a reorder of 127 units now to prevent running out.
  Data:   5.1 days of stock left, reorder_qty 127, current_stock 132, forecast 25.9/day

Store 5, item 27  |  Reorder now  |  Urgency: High
  Reason: There are only 5.4 days of stock left based on current daily demand, so you should reorder 66 units now.
  Data:   5.4 days of stock left, reorder_qty 66, current_stock 75, forecast 14.0/day

Store 3, item 36  |  Reorder now  |  Urgency: High
  Reason: Stock will run out in 5.4 days at a daily predicted demand of 72.6 units, necessitating an immediate reorder of 340 units.
  Dat

## Step 10: Recalculate urgency in Python (no API call)

Urgency is pure arithmetic on numbers we already have, so we don't need the AI for it, and we don't
need to call the Gemini API again. This cell applies the Step 3 thresholds directly to every row of
`data/ai_recommendations.csv`, then saves the corrected file. The AI's `action` and `reason` are kept
as they are.

The rules are applied to the numbers exactly as they appear in the table (rounded to one decimal), so
anyone checking a row by hand gets the same answer.

We also update the urgency in the checkpoint file (`ai_recommendations_partial.csv`). Steps 6 to 8 rebuild
`ai_recommendations.csv` from the checkpoint, so without this, re-running the notebook would bring the old
urgency values back.

In [31]:
def calculate_urgency(row):
    """Apply the Step 3 urgency thresholds to one row of the recommendations table."""
    if row["action"] == "Reorder now":
        days_left = row["days_of_stock_left"]
        if days_left < STOCKOUT_HIGH_BELOW:
            return "High"
        if days_left < STOCKOUT_MEDIUM_BELOW:
            return "Medium"
        return "Low"

    # Overstock: compare stock with the demand forecast for the next 14 days
    next_14_days_demand = row["avg_daily_predicted_demand"] * FORECAST_DAYS
    stock_ratio = row["current_stock"] / next_14_days_demand
    if stock_ratio > OVERSTOCK_HIGH_RATIO:
        return "High"
    if stock_ratio >= OVERSTOCK_MEDIUM_RATIO:
        return "Medium"
    return "Low"


recommendations_path = "../data/ai_recommendations.csv"
recalculated_df = pd.read_csv(recommendations_path)
old_urgency = recalculated_df["urgency"].copy()

# .apply(function, axis=1) runs the function once per ROW (axis=1), passing that row in
recalculated_df["urgency"] = recalculated_df.apply(calculate_urgency, axis=1)

# Sort most urgent first again, with the fewest days of stock first within each level
recalculated_df["urgency"] = pd.Categorical(recalculated_df["urgency"], categories=["High", "Medium", "Low"], ordered=True)
recalculated_df = recalculated_df.sort_values(["urgency", "days_of_stock_left"]).reset_index(drop=True)
recalculated_df.to_csv(recommendations_path, index=False)

# Keep the checkpoint file in step, matching rows by store and item
checkpoint_df = pd.read_csv(PARTIAL_PATH).drop(columns="urgency")
checkpoint_df = checkpoint_df.merge(recalculated_df[["store", "item", "urgency"]], on=["store", "item"], how="left")
checkpoint_df[RECOMMENDATION_COLUMNS].to_csv(PARTIAL_PATH, index=False)

print(f"Previous urgency (rule-based, old threshold):      {old_urgency.value_counts().to_dict()}")
# .value_counts(sort=False) keeps the High, Medium, Low order, including levels with 0 rows
print(f"New urgency (rule-based, threshold = {STOCKOUT_HIGH_BELOW} days): "
      f"{recalculated_df['urgency'].value_counts(sort=False).to_dict()}")
print("\nNew urgency by action:")
# pd.crosstab() counts how often each combination of two columns occurs
print(pd.crosstab(recalculated_df["action"], recalculated_df["urgency"]).to_string())
print(f"\nSaved {len(recalculated_df)} rows to {recommendations_path} (and updated {PARTIAL_PATH})")

Previous urgency (rule-based, old threshold):      {'Medium': 40, 'Low': 26, 'High': 6}
New urgency (rule-based, threshold = 5.5 days): {'High': 6, 'Medium': 40, 'Low': 26}

New urgency by action:
urgency               High  Medium  Low
action                                 
Reduce future orders     0      11    0
Reorder now              6      29   26

Saved 72 rows to ../data/ai_recommendations.csv (and updated ../data/ai_recommendations_partial.csv)
